In [2]:
%cd /content/
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt

import torch
from IPython.display import Image, clear_output

[WinError 2] 지정된 파일을 찾을 수 없습니다: '/content/'
c:\project\Capstone-Project\Lab\Lab01_Object_Dection\yolov5
c:\project\Capstone-Project\Lab\Lab01_Object_Dection\yolov5\yolov5


Cloning into 'yolov5'...


^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not install packages due to an OSError: [WinError 5] 액세스가 거부되었습니다: 'C:\\Users\\wnsgu\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip available: 22.3.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


### 데이터셋 다운로드

* 데이터셋: https://universe.roboflow.com/brailledataset/braille_dataset/dataset/2

In [ ]:
%mkdir /content/yolov5/braille
%cd /content/yolov5/braille

curl -L "https://universe.roboflow.com/ds/btKyGU4MaG?key=eCC4eqr7Wu" > roboflow.zip; unzip roboflow.zip; rm roboflow.zip

In [2]:
from glob import glob

train_img_list = glob('./dataset/train/images/*.jpg')
test_img_list = glob('./dataset/test/images/*.jpg')
valid_img_list = glob('./dataset/valid/images/*.jpg')
print(len(train_img_list), len(test_img_list), len(valid_img_list))

282 21 38


In [ ]:
import yaml

with open('./dataset/train.txt', 'w') as f:
  f.write('\n'.join(train_img_list) + '\n')

with open('./dataset/test.txt', 'w') as f:
  f.write('\n'.join(test_img_list) + '\n')

with open('./dataset/val.txt', 'w') as f:
  f.write('\n'.join(valid_img_list) + '\n')

In [ ]:
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def writetemplate(line,cell):
  with open(line, 'w') as f:
    f.write(cell.format(**globals()))

In [ ]:
%cat /content/yolov5/baille/data.yaml

In [ ]:
%%writetemplate /content/yolov5/baille/data.yaml

train: ./baille/train/images
test: ./baille/test/images
val: ./baille/valid/images

nc: 64
names: ['']

In [ ]:
%cat /content/yolov5/baille/data.yaml

### 모델 구성

In [ ]:
import yaml

with open('/content/yolov5/pothole/data.yaml', 'r') as stream:
  num_classes = str(yaml.safe_load(stream)['nc'])

%cat /content/yolov5/models/yolov5s.yaml

In [ ]:
%%writetemplate /content/yolov5/models/custom_yolov5s.yaml


# YOLOv5 🚀 by Ultralytics, AGPL-3.0 license

# Parameters
nc: {num_classes} # number of classes
depth_multiple: 0.33 # model depth multiple
width_multiple: 0.50 # layer channel multiple
anchors:
  - [10, 13, 16, 30, 33, 23] # P3/8
  - [30, 61, 62, 45, 59, 119] # P4/16
  - [116, 90, 156, 198, 373, 326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [
    [-1, 1, Conv, [64, 6, 2, 2]], # 0-P1/2
    [-1, 1, Conv, [128, 3, 2]], # 1-P2/4
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]], # 3-P3/8
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]], # 5-P4/16
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]], # 7-P5/32
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]], # 9
  ]

# YOLOv5 v6.0 head
head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]], # cat backbone P4
    [-1, 3, C3, [512, False]], # 13

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]], # cat backbone P3
    [-1, 3, C3, [256, False]], # 17 (P3/8-small)

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]], # cat head P4
    [-1, 3, C3, [512, False]], # 20 (P4/16-medium)

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]], # cat head P5
    [-1, 3, C3, [1024, False]], # 23 (P5/32-large)

    [[17, 20, 23], 1, Detect, [nc, anchors]], # Detect(P3, P4, P5)
  ]

In [ ]:
%cat /content/yolov5/models/custom_yolov5s.yaml

### 학습(Training)

* `img`: 입력 이미지 크기 정의
* `batch`: 배치 크기 결정
* `epochs`: 학습 기간 개수 정의
* `data`: yaml 파일 경로
* `cfg`: 모델 구성 지정
* `weights`: 가중치에 대한 경로 지정
* `name`: 결과 이름
* `nosave`: 최종 체크포인트만 저장
* `cache`: 빠른 학습을 위한 이미지 캐시

In [ ]:
%%time
%cd /content/yolov5/
!python train.py --img 640 --batch 32 --epochs 200 --data ./baille/data.yaml --cfg ./models/custom_yolov5s.yaml --weights '' --name baille_results --cache

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
!ls /content/yolov5/runs/train/baille/

In [ ]:
Image(filename='/content/yolov5/runs/train/baille_results/results.png', width=1000)

In [ ]:
Image(filename='/content/yolov5/runs/train/baille_results/train_batch0.jpg', width=1000)

In [ ]:
Image(filename='/content/yolov5/runs/train/baille_results/val_batch0_labels.jpg', width=1000)

### 검증(Validation)

In [ ]:
!python val.py --weights runs/train/baille_results/weights/best.pt --data ./baille/data.yaml --img 640 --iou 0.65 --half

In [ ]:
!python val.py --weights runs/train/baille_results/weights/best.pt --data ./baille/data.yaml --img 640 --task test

### 추론(Inference)

In [ ]:
%ls runs/train/pothole_results/weights

In [ ]:
!python detect.py --weights runs/train/baille_results/weights/best.pt --img 640 --conf 0.4 --source ./baille/test/images

In [ ]:
import glob
import random
from IPython.display import Image,display

image_name = random.choice(glob.glob('/content/yolov5/runs/detect/exp2/*.jpg'))
display(Image(filename=image_name))

### 모델 내보내기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%mkdir /content/drive/My\ Drive/baille
%cp /content/yolov5/runs/train/baille_results/weights/best.pt /content/drive/My\ Drive/baille